Copyright (c) Qualcomm Technologies, Inc. and/or its subsidiaries. <br>
SPDX-License-Identifier: BSD-3-Clause-Clear

**Takeaways** : Users will learn to profile inferences running on Cloud AI 100 at a system and device level <br><br>
**Before you start** : Some commands (folder locations etc) will need to be updated based on the platform and installation locations. 'Sudo' prefix might be required for some commands to run properly. <br> This notebook assumes that the user understands how to onboard a model, compile and run using qaic-compile and qaic-runner (AIC native tools) respectively. <br>
The terms 'model' and 'network' are used interchangeably in this notebook. 
The terms 'NSP' and 'AI compute core" are used interchangeably in this notebook. 

**Last Verified Qualcomm Cloud AI Platform SDK and Apps SDK Version:** Platform SDK 1.23.2.0 and Apps SDK 1.23.2.0 

# <span style='color:Blue'> Profiling Inferences </span>

##  Pre-requisite reading 
New users on Cloud AI platforms are expected to have a good understanding of the compilation (qaic-compile) and inference (qaic-runner) execution tools. 

## Introduction 
This notebook will cover inference profiling at system and device level using the 'bert-base-cased' model from HuggingFace. 

Here is the workflow that will be demonstrated in this notebook. 

1. **Install required packages and download the model**: Begin by installing the required packages and downloading the model.  
2. **Device health check**: Query the device health using qaic-util tool. 
3. **Device-level profiling** :
    - Compiler estimation of the resources used on device. 
    - Summary of the operators, its count and device cycles used. 
    - Detailed timeline view of inference execution on device
4. **System-level profiling**: Breakdown the inference time at system level - application, linux runtime, kernel and device.  



# <span style='color:Blue'> 1. Install required packages and download the model </span>

We will install the required Python packages 

In [ ]:
import sys
sys.executable

In [ ]:
!pip install -r requirements.txt

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Recent transformers versions default to SDPA attention, which cannot be exported
# to opset 11 (aten::scaled_dot_product_attention needs opset 14). Load the
# model with eager attention and export the saved local copy instead.
from transformers import AutoModelForQuestionAnswering
import json
model = AutoModelForQuestionAnswering.from_pretrained("bert-base-cased", attn_implementation="eager")
model.save_pretrained("bert_base_cased_eager/")
cfg_path = "bert_base_cased_eager/config.json"
cfg = json.load(open(cfg_path))
cfg["_attn_implementation"] = "eager"  # save_pretrained drops this private attribute; re-inject so optimum-cli exports the eager graph
json.dump(cfg, open(cfg_path, "w"), indent=2)
!optimum-cli export onnx --model bert_base_cased_eager --cache_dir model_files/cased --opset 11 --task question-answering bert_base_cased_onnx/

In [ ]:
## Clip the constant (used to represent -inf) out of FP16 range to FP16.   

import numpy as np
import onnx
from onnx import numpy_helper
        
def fix_onnx_fp16(
    gen_models_path: str,
    model_base_name: str,
) -> str:
    finfo = np.finfo(np.float16)
    fp16_max = finfo.max
    fp16_min = finfo.min

    model = onnx.load(f"{gen_models_path}/{model_base_name}.onnx")
    fp16_fix = False
    for tensor in onnx.external_data_helper._get_all_tensors(model):
        nptensor = numpy_helper.to_array(tensor, gen_models_path)
        if nptensor.dtype == np.float32 and (
            np.any(nptensor > fp16_max) or np.any(nptensor < fp16_min)
        ):
            print(f'tensor value : {nptensor} above {fp16_max} or below {fp16_min}')
            nptensor = np.clip(nptensor, fp16_min, fp16_max)
            new_tensor = numpy_helper.from_array(nptensor, tensor.name)
            tensor.CopyFrom(new_tensor)
            fp16_fix = True
            
    if fp16_fix:
        # Save FP16 model
        print("Found constants out of FP16 range, clipped to FP16 range")
        model_base_name += "_fix_outofrange_fp16"
        onnx.save(model, f=f"{gen_models_path}/{model_base_name}.onnx")
        print(f"Saving modified onnx file at {gen_models_path}/{model_base_name}.onnx")
    return model_base_name

fp16_model_name = fix_onnx_fp16(gen_models_path="bert_base_cased_onnx", model_base_name="model")

# <span style='color:Blue'>2. Check Device Health </span>

'qaic-util' tool can be used to query the health of all the Cloud AI cards in the server. 

"Status: Ready" indicates that the health of the cards is good. 

"Status: Error" indicates that the cards are not in good health and a system administrator needs to be contacted to rectify the issue. 


In [ ]:
!/opt/qti-aic/tools/qaic-util -q | grep "Status"

# <span style='color:Blue'>3. Device Level Profiling - Compiler Opstats </span>

The Compiler Operator Trace (OpStat) feature allows capturing of network operator trace information from the NSP AI cores combined with QAic Linux Runtime and QAic Kernel events from the host. The combined trace shows a timeline view of host processing along with operator execution and synchronization delays. To enable operator traces, the following parameters need to be added. The number of samples aic-profiling-num-samples and starting iteration aic-profiling-start-iter can be adjusted. The profiling output directory aic-profiling-out-dir must exist.

**qaic-compile**: -aic-perf-metrics
-aic-perf-metrics : prints an outline of the resource usage of the compiled network

**qaic-runner**: aic-profiling-type : latency
aic-profiling-out-dir : output directory for latency capture (needs to exist before this command is run)
aic-profiling-start-iter : Set this value high enough to start capturing samples after device warmup (~250mS)
aic-profiling-num-samples : # of samples to be captured. Can be set greater than # of inferences

**Sequence of Steps to be run**

1. Compile with the performance metrics flag
2. Run with the profiling flags



In [ ]:
!rm -rf compiled_fp16
!/opt/qti-aic/exec/qaic-compile -v -aic-hw  -convert-to-fp16 \
-mos=1 -ols=1 -aic-num-cores=2 \
-m=bert_base_cased_onnx/{fp16_model_name}.onnx \
-onnx-define-symbol=sequence_length,128 -onnx-define-symbol=batch_size,8 \
-aic-perf-metrics -multicast-weights -stats-batchsize=8 -aic-binary-dir=./compiled_fp16 \
-aic-hw-version=ai100

The runner's `raw_device_stats` profiling type only collects data if the QPC has operator statistics baked in at compile time. Compile a separate diagnostic binary for that purpose — a QPC compiled with `-stats-level` should never be used for the throughput/latency benchmarks below. At execution, the raw device-stats buffers are written to the **runner's working directory** (the `--aic-profiling-out-dir` receives the latency/trace files), and `-ddr-stats` QPCs also print `DDRTrafficMB` / `DDRBandwidthGBPerSec` sample lines to stdout.

In [ ]:
## Compile the diagnostic binary (stats-level bakes operator stats into the QPC)
!rm -rf compiled_fp16_diag
!/opt/qti-aic/exec/qaic-compile -v -aic-hw  -convert-to-fp16 \
-mos=1 -ols=1 -aic-num-cores=2 \
-m=bert_base_cased_onnx/{fp16_model_name}.onnx \
-onnx-define-symbol=sequence_length,128 -onnx-define-symbol=batch_size,8 \
-stats-level=80 -ddr-stats -multicast-weights -stats-batchsize=8 -aic-binary-dir=./compiled_fp16_diag \
-aic-hw-version=ai100

**-aic-perf-metrics** dumps the compiler estimate of the usage of DDR vs VTCM (Vector Tightly Coupled Memory). The compiler tries to place the intermediate layer outputs and weights in VTCM. 'Constant only' refer to the weights of the network. The difference in 'DDR traffic in' and 'constant only' provides any spilover of intermediate outputs to DDR which impacts performance.


In [ ]:
!mkdir -p bert_base_cased_stats
!cd bert_base_cased_stats && /opt/qti-aic/exec/qaic-runner --test-data ../compiled_fp16_diag  -d 0 -S 1 -a 7 \
--aic-profiling-type latency --aic-profiling-out-dir . \
--aic-profiling-type raw_device_stats --aic-profiling-type trace \
--aic-profiling-start-iter 100 --aic-profiling-num-samples 1 --time 20



## Summary View 
A summary text file is created per instance (running on 1 or more NSP cores) showing distribution of the load on every NSP. There are 4 HVX(Vector Unit), 1 HMX(Matrix Unit) and 1 DMA engine per NSP. This stats shows % utilization of every engine with total number of cycles required for computation. It also has operator wise break down of cycles consumed.

In [ ]:
## Post process the raw device stats (files carry the 1-based iteration of the sample: inf-101 for start-iter 100)
!mkdir -p opstats_summary
!/opt/qti-aic/exec/qaic-opstats -q ./compiled_fp16_diag/programqpc.bin -i ./bert_base_cased_stats --summary -o ./opstats_summary
!ls opstats_summary
!cat opstats_summary/*summary* 2>/dev/null || cat opstats_summary/*

## Detailed Timeline View
JSON files hold the detailed timeline view of the inference executed on device. 
Open chrome://tracing in Chrome/Chromium and click the Load button to open and view one of the JSON files. Use the toolbar to select, pan and zoom the timelines.

**Reading trace timeline** <br>
The trace timeline is divided into HOST_PROC and Process 0 .. N sections.

<div>
<img src="images/opstats_decoder.png" width="500"/>
</div>

**Viewing Operators** <br>
Zoom in to see individual operations. Here's an example showing convolution and softmax operations.

<div>
<img src="images/opstats_example.png" width="1000"/>
</div>

**Viewing Operator details** <br>
Click on an operator to see more details. Here are details of one of the softmax operations. 

<div>
<img src="images/operator_details.png" width="1000"/>
</div>

# <span style='color:Blue'>4. System Level Profiling </span>

System level profiling includes the breakdown of the inference time between Application, linux runtime , kernel mode driver and hardware processing.

qaic-runner tools can help with system level profiling. The following flags are required during inference. 

**qaic-runner** <br>
aic-profiling-type : latency <br>
aic-profiling-out-dir : output directory for latency capture (needs to exist before this command is run) <br>
aic-profiling-start-iter : Set this value high enough to start capturing samples after device warmup (~250mS) <br>
aic-profiling-num-samples : # of samples to be captured. Can be set greater than # of inferences <br><br>

<div>
<img src="images/Latency.png" width="1000"/>
</div>

In [ ]:
!/opt/qti-aic/exec/qaic-runner --test-data ./compiled_fp16  -d 0 -S 1 -a 7 \
--aic-profiling-type latency --aic-profiling-out-dir ./bert_base_cased_stats \
--aic-profiling-start-iter 100 --aic-profiling-num-samples 99999 --time 20 

In [ ]:
!python3 latency_stats_python3.py ./bert_base_cased_stats/aic-profiling-program-0-latency.txt 